# 🦀 Day 7: Performance Tuning
---

Today we profile and optimize RustKV.

- **Profiling**: Where are the bottlenecks?
- **Benchmarking**: criterion for ops/second (GET, SET, DELETE)
- **Memory optimization**: Reduce allocations in hot paths
- **Lock contention**: RwLock read-heavy optimization; sharded storage
- **Connection pooling**: Client-side pooling
- **Batch operations**: Pipeline multiple commands
- **Exercise**: Implement command pipelining

## Sharded storage — reduce lock contention

Split the HashMap into N shards. Each shard has its own RwLock. Key hash determines shard — fewer locks contended.

In [ ]:
use std::collections::HashMap;
use std::hash::{Hash, Hasher};
use std::sync::RwLock;

const NUM_SHARDS: usize = 64;

pub struct ShardedStorage {
    shards: [RwLock<HashMap<String, String>>; NUM_SHARDS],
}

fn shard_index(key: &str) -> usize {
    let mut h = std::collections::hash_map::DefaultHasher::new();
    key.hash(&mut h);
    (h.finish() as usize) % NUM_SHARDS
}

impl ShardedStorage {
    pub fn new() -> Self {
        Self {
            shards: std::array::from_fn(|_| RwLock::new(HashMap::new())),
        }
    }

    pub fn get(&self, key: &str) -> Option<String> {
        let idx = shard_index(key);
        self.shards[idx].read().unwrap().get(key).cloned()
    }

    pub fn set(&self, key: &str, value: &str) {
        let idx = shard_index(key);
        self.shards[idx].write().unwrap().insert(key.to_string(), value.to_string());
    }

    pub fn delete(&self, key: &str) -> bool {
        let idx = shard_index(key);
        self.shards[idx].write().unwrap().remove(key).is_some()
    }
}

let s = ShardedStorage::new();
s.set("foo", "bar");
println!("Sharded: get(foo) = {:?}", s.get("foo"));

In [ ]:
// Benchmark harness — criterion for GET/SET/DELETE

let bench_code = r#"
// In Cargo.toml:
// [dev-dependencies]
// criterion = { version = "0.5", features = ["html_reports"] }

// benches/ops_bench.rs:
use criterion::{black_box, criterion_group, criterion_main, Criterion};

fn bench_set(c: &mut Criterion) {
    let storage = ShardedStorage::new();
    c.bench_function("set", |b| {
        b.iter(|| storage.set(black_box("key"), black_box("value")))
    });
}

fn bench_get(c: &mut Criterion) {
    let storage = ShardedStorage::new();
    storage.set("key", "value");
    c.bench_function("get", |b| {
        b.iter(|| storage.get(black_box("key")))
    });
}

criterion_group!(benches, bench_set, bench_get);
criterion_main!(benches);

// Run: cargo bench
"#;

println!("Benchmark harness:");
println!("{}", bench_code);

In [ ]:
// Memory optimization — reduce allocations in hot paths

let mem_tips = r#"
1. Reuse buffers: Vec::clear() + extend instead of allocating new Vec each request
2. Avoid String::from in hot path: use &str or Cow where possible
3. Pre-allocate: Vec::with_capacity(n) when size known
4. Use Bytes or BytesMut for protocol buffers — zero-copy where possible
5. Consider Arc<str> for keys if cloned often
"#;

println!("Memory tips:");
println!("{}", mem_tips);

In [ ]:
// Command pipelining — send multiple commands without waiting for each response

let pipeline = r#"
// Client sends: SET a 1\nSET b 2\nGET a\nGET b\n
// Server processes in order, sends responses in order
// Client reads N responses when N commands were sent
// Benefit: lower latency when issuing many commands (e.g., batch load)

// Implementation: queue outgoing commands; have a reader task that matches
// responses to requests by order. Or use Redis-style pipeline where
// client buffers commands and flushes; then reads all responses.
"#;

println!("Pipeline concept:");
println!("{}", pipeline);

## 📝 Exercise: Implement command pipelining

On the client: allow sending multiple commands in one batch (e.g., `SET a 1; SET b 2; GET a`), then read all responses. On the server: ensure responses are sent in the same order as commands received.

In [ ]:
// YOUR CODE HERE
// Client: split input by ';', send each command, then read N responses
// Server: already processes in order; just ensure write_all per response

// let commands: Vec<&str> = input.split(';').map(str::trim).filter(|s| !s.is_empty()).collect();
// for cmd in &commands { stream.write_all(cmd.as_bytes())?; stream.write_all(b"\n")?; }
// for _ in &commands { let mut buf = String::new(); rdr.read_line(&mut buf)?; println!("{}", buf.trim()); }

println!("Implement command pipelining on client.");

## 🎯 Key Takeaways

1. **Sharded storage**: Split HashMap into N shards to reduce RwLock contention — big win for concurrent workloads
2. **Benchmarking**: Use criterion for GET/SET/DELETE ops/sec; compare before/after optimizations
3. **Memory**: Reuse buffers, avoid allocations in hot paths, pre-allocate when size known
4. **Connection pooling**: Client reuses connections for multiple requests — reduces connect overhead
5. **Pipelining**: Send multiple commands, read multiple responses — lowers latency for batch operations

---
**Week 22 Complete!** RustKV now has networking, data structures, pub/sub, tests, CLI, and performance tuning. Next week: polish & open source!